# Exploratory Analysis — PokerBeliefStudy

This notebook loads hand records, computes performance summaries, and provides initial exploration of experiment results.

In [ ]:
import os
import sys
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to path
sys.path.insert(0, os.path.abspath('..'))

from src.analysis import (
    compute_performance_summary,
    compute_calibration_metrics,
    compute_robustness_metrics,
    plot_performance_comparison,
    plot_reliability_diagram,
    plot_robustness_heatmap,
    plot_belief_trace,
)

# Plot settings
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')

# Directories
OUTPUT_DIR = '../outputs'
RAW_DIR = os.path.join(OUTPUT_DIR, 'raw_runs')
PROCESSED_DIR = os.path.join(OUTPUT_DIR, 'processed')
FIGURES_DIR = os.path.join(OUTPUT_DIR, 'figures')

print('Setup complete.')

## 1. Loading Hand Records

In [ ]:
def load_all_summaries(processed_dir):
    """Load all processed CSV summaries."""
    dfs = {}
    if not os.path.exists(processed_dir):
        print(f'Processed directory not found: {processed_dir}')
        return dfs
    for fname in os.listdir(processed_dir):
        if fname.endswith('_summary.csv'):
            exp_id = fname.replace('_summary.csv', '')
            df = pd.read_csv(os.path.join(processed_dir, fname))
            dfs[exp_id] = df
            print(f'Loaded {exp_id}: {len(df):,} hands')
    return dfs

def load_raw_records(raw_dir, limit=None):
    """Load raw JSON hand records."""
    records = []
    if not os.path.exists(raw_dir):
        print(f'Raw directory not found: {raw_dir}')
        return records
    files = [f for f in os.listdir(raw_dir) if f.endswith('.json')]
    for fname in files[:limit] if limit else files:
        fpath = os.path.join(raw_dir, fname)
        with open(fpath) as f:
            batch = json.load(f)
        records.extend(batch)
    print(f'Loaded {len(records):,} hand records from {len(files)} files')
    return records

all_dfs = load_all_summaries(PROCESSED_DIR)
raw_records = load_raw_records(RAW_DIR, limit=10)

print(f'\nAvailable experiments: {list(all_dfs.keys())}')

## 2. Performance Summary

In [ ]:
# Compute performance summary for each experiment
for exp_id, df in all_dfs.items():
    print(f'\n=== {exp_id} ===')
    print(f'Total hands: {len(df):,}')
    
    if 'agent0_type' in df.columns:
        print('\nMean reward by agent (player 0):')
        for agent_type, grp in df.groupby('agent0_type'):
            mean_r = grp['terminal_reward_0'].mean()
            std_r = grp['terminal_reward_0'].std()
            n = len(grp)
            print(f'  {agent_type:20s}: {mean_r:8.2f} ± {std_r:.2f} (n={n:,})')
    
    if 'realized_hand_class_0' in df.columns:
        print('\nHand class distribution (player 0):')
        vc = df['realized_hand_class_0'].value_counts(normalize=True)
        for cls, freq in vc.items():
            print(f'  {cls:25s}: {freq:.3f}')

## 3. Chips per Hand Comparison

In [ ]:
# Plot performance comparison for main_comparison experiment
if 'main_comparison' in all_dfs:
    df = all_dfs['main_comparison']
    summary = compute_performance_summary(df)
    
    if not summary.empty:
        fig, ax = plt.subplots(figsize=(10, 6))
        
        agents = summary['agent'].unique()
        x = np.arange(len(agents))
        colors = sns.color_palette('colorblind', len(agents))
        
        means = [summary[summary['agent']==a]['mean_reward'].mean() for a in agents]
        lowers = [summary[summary['agent']==a]['ci_lower'].mean() for a in agents]
        uppers = [summary[summary['agent']==a]['ci_upper'].mean() for a in agents]
        
        bars = ax.bar(x, means, 0.6, color=colors, alpha=0.8, edgecolor='black')
        ax.errorbar(x, means, 
                    yerr=[[m-l for m,l in zip(means,lowers)], 
                           [u-m for m,u in zip(means,uppers)]],
                    fmt='none', color='black', capsize=5)
        
        ax.set_xticks(x)
        ax.set_xticklabels(agents, fontsize=11)
        ax.set_ylabel('Mean Chips per Hand', fontsize=12)
        ax.set_title('Agent Performance Comparison', fontsize=14, fontweight='bold')
        ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
        
        plt.tight_layout()
        plt.show()
    else:
        print('No summary data available.')
else:
    print('main_comparison experiment not found. Run it first:')
    print('  python run_experiment.py --experiment main_comparison --seeds 42,43 --hands 100')

## 4. Calibration Analysis

In [ ]:
if raw_records:
    # Filter to belief agent records (have posterior data)
    belief_records = [
        r for r in raw_records 
        if any(step.get('posterior') is not None 
               for step in r.get('action_history', []))
    ]
    
    print(f'Records with belief data: {len(belief_records)}/{len(raw_records)}')
    
    if belief_records:
        cal_data = compute_calibration_metrics(belief_records)
        
        print(f'\nCalibration Metrics:')
        print(f'  Brier Score: {cal_data["brier_score"]}')
        print(f'  ECE: {cal_data["ece"]}')
        print(f'  N predictions: {cal_data["n_predictions"]}')
        
        if cal_data['reliability_data']:
            rel = cal_data['reliability_data']
            fig, ax = plt.subplots(figsize=(6, 6))
            ax.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
            ax.plot(rel['mean_confidence'], rel['mean_accuracy'], 
                    'o-', color='steelblue', label='Model')
            ax.set_xlabel('Confidence')
            ax.set_ylabel('Accuracy')
            ax.set_title('Reliability Diagram')
            ax.legend()
            plt.tight_layout()
            plt.show()
    else:
        print('No belief data found in raw records.')
else:
    print('No raw records loaded. Run an experiment first.')

## 5. Robustness Analysis

In [ ]:
if 'robustness' in all_dfs:
    df = all_dfs['robustness']
    rob = compute_robustness_metrics(df)
    
    print('Robustness Metrics:')
    if not rob.empty:
        display(rob.round(3))
        
        fig, ax = plt.subplots(figsize=(8, 4))
        x = np.arange(len(rob))
        width = 0.35
        
        bars1 = ax.bar(x - width/2, rob['dev_mean_reward'], width, label='Dev Opponents', alpha=0.8)
        bars2 = ax.bar(x + width/2, rob['held_out_mean_reward'], width, label='Held-Out Opponents', alpha=0.8)
        
        ax.set_xticks(x)
        ax.set_xticklabels(rob['agent'], fontsize=10)
        ax.set_ylabel('Mean Chips per Hand')
        ax.set_title('Robustness: Dev vs Held-Out Opponents')
        ax.legend()
        ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
        
        plt.tight_layout()
        plt.show()
    else:
        print('Empty robustness data.')
else:
    print('Robustness experiment not found. Run:')
    print('  python run_experiment.py --experiment robustness --seeds 42,43 --hands 100')

## 6. Action Frequency Analysis

In [ ]:
if all_dfs:
    # Use first available experiment
    exp_id = list(all_dfs.keys())[0]
    df = all_dfs[exp_id]
    
    # Load corresponding raw records to get action frequencies
    raw_files = [f for f in os.listdir(RAW_DIR) if f.startswith(exp_id) and f.endswith('.json')]
    
    if raw_files:
        all_actions = []
        for fname in raw_files[:3]:
            with open(os.path.join(RAW_DIR, fname)) as f:
                records = json.load(f)
            for rec in records:
                for step in rec.get('action_history', []):
                    all_actions.append({
                        'action': step['action'],
                        'player': step['player'],
                        'street': step['street'],
                    })
        
        if all_actions:
            action_df = pd.DataFrame(all_actions)
            
            fig, axes = plt.subplots(1, 2, figsize=(12, 4))
            
            # Action frequency by street
            for i, street in enumerate(['turn', 'river']):
                street_df = action_df[action_df['street'] == street]
                if len(street_df) > 0:
                    vc = street_df['action'].value_counts()
                    vc.plot(kind='bar', ax=axes[i], color=sns.color_palette('colorblind', len(vc)))
                    axes[i].set_title(f'Action Frequency ({street.capitalize()})')
                    axes[i].set_xlabel('Action')
                    axes[i].set_ylabel('Count')
                    axes[i].tick_params(axis='x', rotation=45)
            
            plt.tight_layout()
            plt.show()
            
            print(f'Total actions analyzed: {len(action_df):,}')
    else:
        print('No raw files found for action analysis.')
else:
    print('No experiment data available.')

## 7. Belief Trace Example

In [ ]:
# Find a hand with belief data and plot the trace
belief_hand = None
for rec in raw_records:
    has_belief = any(
        step.get('posterior') is not None 
        for step in rec.get('action_history', [])
    )
    if has_belief:
        belief_hand = rec
        break

if belief_hand:
    print(f"Hand ID: {belief_hand.get('hand_id', 'N/A')}")
    print(f"Board: {' '.join(belief_hand['board'])}")
    print(f"Hole cards (P0): {' '.join(belief_hand['hole_cards_0'])}")
    print(f"Winner: {belief_hand.get('showdown_winner', 'N/A')}")
    print(f"Reward P0: {belief_hand.get('terminal_reward_0', 0)}")
    
    from src.hand_classes import HAND_CLASSES
    
    # Extract belief trace
    steps = [s for s in belief_hand['action_history'] if s.get('posterior')]
    
    if steps:
        fig, ax = plt.subplots(figsize=(10, 5))
        colors = sns.color_palette('colorblind', len(HAND_CLASSES))
        
        x = [s['decision_index'] for s in steps]
        for i, hc in enumerate(HAND_CLASSES):
            probs = [s['posterior'].get(hc, 0) for s in steps]
            ax.plot(x, probs, 'o-', color=colors[i], label=hc, linewidth=2, markersize=5)
        
        ax.set_xlabel('Decision Index')
        ax.set_ylabel('Posterior Probability')
        ax.set_title(f"Belief Evolution — Board: {' '.join(belief_hand['board'])}")
        ax.legend(fontsize=8, ncol=2)
        ax.set_ylim([0, 1])
        
        plt.tight_layout()
        plt.show()
    else:
        print('No belief steps with posterior data.')
else:
    print('No hand with belief data found. Run the belief_ablation or calibration experiment.')